# fair-prep — Colab / Local runner

End-to-end pipeline: SFT, DPO, cross-size LoRA transfer, merging scaling-law.

**Where it runs**:
- **Colab** (T4/A100/L4): bootstrap cell clones repo + installs unsloth.
- **Local VS Code** (Mac MPS / Linux CUDA): bootstrap detects existing checkout, skips clone. Pick the `.venv` kernel from VS Code top-right.

Device auto-detected by `lb` (CUDA > MPS > CPU).

## 1 · Bootstrap (Colab clone + install, or local detect)

In [1]:
import os, sys, subprocess, pathlib

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
print('IN_COLAB =', IN_COLAB)

if IN_COLAB:
    os.environ.setdefault('WITH_UNSLOTH', '1')
    # os.environ['HF_TOKEN'] = 'hf_xxx'  # uncomment for gated models
    subprocess.run(
        'bash <(curl -sL https://raw.githubusercontent.com/Shumatsurontek/fair-prep/main/colab/setup.sh)',
        shell=True, executable='/bin/bash', check=True,
    )
    os.chdir('/content/fair-prep/lora-bench')
else:
    # Local: walk up from the notebook to find lora-bench/.
    here = pathlib.Path.cwd()
    for p in [here, *here.parents]:
        if (p / 'lora-bench' / 'lb').exists():
            os.chdir(p / 'lora-bench')
            break
    else:
        raise RuntimeError('lora-bench/ not found above ' + str(here))

import torch
print('cwd        =', os.getcwd())
print('torch      =', torch.__version__)
print('cuda avail =', torch.cuda.is_available())
print('mps  avail =', torch.backends.mps.is_available())

IN_COLAB = True
cwd        = /content/fair-prep/lora-bench
torch      = 2.10.0+cu128
cuda avail = True
mps  avail = False


In [2]:
!./lb status
!./lb models

                  runs/                   
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ name                ┃ mtime            ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ qwen3_06b_sft_gsm8k │ 2026-05-12 14:45 │
│ teacher_4b          │ 2026-05-12 14:49 │
│ transferred         │ 2026-05-12 14:53 │
└─────────────────────┴──────────────────┘
                   results/                    
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━┓
┃ file                          ┃   acc ┃   n ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━┩
│ baseline.json                 │ 0.220 │ 200 │
│ dpo.json                      │ 0.370 │ 200 │
│ math500_baseline.json         │ 0.225 │ 200 │
│ math500_baseline_n500.json    │ 0.180 │ 500 │
│ math500_dpo.json              │ 0.280 │ 200 │
│ math500_dpo_n500.json         │ 0.264 │ 500 │
│ math500_gtw.json              │ 0.265 │ 200 │
│ math500_gtw_n500.json         │ 0.234 │ 500 │
│ math500_sft.json              │ 0.265 │ 200 │
│ math500_sft_n500.json         

## 2 · Baseline eval (no adapter)

Tip: lower `-n` (e.g. `-n 20`) for a quick local smoke test on MPS.

In [ ]:
N = 200 if torch.cuda.is_available() else 20
!./lb eval gsm8k -M qwen3-0.6b -n 200 -B 16 --max-new-tokens 256
!./lb eval math500 -M qwen3-0.6b -n 200 -B 16 --max-new-tokens 512


In [4]:
!uv pip install --system --no-deps --force-reinstall "cuda-bindings==12.9.4" "cuda-toolkit==12.8.1"
!uv pip install --system --force-reinstall "transformers==4.46.3" "trl==0.11.4" "peft==0.13.2"


Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 123ms                                         
Prepared 2 packages in 320ms                                             
Uninstalled 2 packages in 11ms
Installed 2 packages in 10ms                                
 ~ cuda-bindings==12.9.4
 ~ cuda-toolkit==12.8.1
Using Python 3.12.13 environment at: /usr
Resolved 71 packages in 301ms                                        
Prepared 71 packages in 27.09s                                           
Uninstalled 56 packages in 1.03s
Installed 71 packages in 622ms                              
 ~ accelerate==1.13.0
 ~ aiohappyeyeballs==2.6.1
 ~ aiohttp==3.13.5
 ~ aiosignal==1.4.0
 ~ anyio==4.13.0
 ~ attrs==26.1.0
 ~ certifi==2026.4.22
 ~ charset-normalizer==3.4.7
 - cuda-bindings==12.9.4
 + cuda-bindings==13.2.0
 - cuda-pathfinder==1.5.3
 + cuda-pathfinder==1.5.4
 - cuda-toolkit==12.8.1
 + cuda-toolkit==13.0.2
 - datasets==4.3.0
 + datasets==4.8.5
 - dill==0.3.8
 + dill==0.4.1
 ~

## 3 · SFT (standard, TRL)

In [4]:
# Small `--max-steps 50 -n 200` for smoke run on local MPS.
# Remove for a full run on CUDA.
STEPS = -1 if torch.cuda.is_available() else 50
TRAIN_N = '' if torch.cuda.is_available() else '-n 200'
!./lb train sft -M qwen3-0.6b -c configs/sft_gsm8k.yaml -s {STEPS} {TRAIN_N}
!./lb eval gsm8k   -a runs/qwen3_06b_sft_gsm8k/final -n {N} -o results/sft.json
!./lb eval math500 -a runs/qwen3_06b_sft_gsm8k/final -n {N} -o results/math500_sft.json

╭────────────── lb ──────────────╮
│ device  cuda                   │
│ model   Qwen/Qwen3-0.6B        │
│ cfg     configs/sft_gsm8k.yaml │
│ lora_r  16                     │
│ run     qwen3_06b_sft_gsm8k    │
╰────────────────────────────────╯
2026-05-12 14:51:41.555364: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Usage: lb eval gsm8k [OPTIONS]
Try 'lb eval gsm8k --help' for help.
╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ Invalid value for '--max-samples' / '-n': '{N}' is not a valid integer.      │
╰──────────────────────────────────────────────────────────────────────────────╯

## 4 · SFT (unsloth fast path — CUDA only)

Auto-falls back to TRL if no CUDA or unsloth not installed.

In [5]:
!./lb train sft -M qwen3-0.6b -c configs/sft_gsm8k.yaml --fast -s {STEPS} {TRAIN_N}

╭────────────── lb ──────────────╮
│ device  cuda                   │
│ model   Qwen/Qwen3-0.6B        │
│ cfg     configs/sft_gsm8k.yaml │
│ lora_r  16                     │
│ run     qwen3_06b_sft_gsm8k    │
╰────────────────────────────────╯
2026-05-12 14:51:58.289879: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
/content/fair-prep/lora-bench/src/trainers/sft_unsloth.py:41: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'impor

## 5 · Cross-size LoRA transfer (research axis)

Teacher (Qwen3-4B) LoRA → student (Qwen3-0.6B) via OLS projection on calibration activations. Layer mapping via linear stride.

In [3]:
# 5.1 download public Qwen3-4B math LoRA (adapter only, ~120 MB)
from huggingface_hub import snapshot_download
snapshot_download(
    'saaduddinM/MATH_SFT_Q3.0-4BI_r16',
    local_dir='runs/teacher_4b/math_sft',
    allow_patterns=['adapter_*', '*.json'],
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

'/content/fair-prep/lora-bench/runs/teacher_4b/math_sft'

In [ ]:
# 5.2 build calibration set from MATH-500 prompts
# CPU RAM = n_calib × max_length × d × bytes × 2 (X,Y) × 7 modules × 28 layers
# 64 × 128 × 2560 × 2 × 2 × 7 × 28 ≈ 8 GB → safe on Colab 12 GB.
import json
from datasets import load_dataset
os.makedirs('data', exist_ok=True)
n_calib = 64
ds = load_dataset('HuggingFaceH4/MATH-500', split='test').shuffle(seed=42).select(range(n_calib))
with open('data/calib_math.jsonl', 'w') as f:
    for ex in ds:
        f.write(json.dumps({'text': ex['problem']}) + '\n')
print(f'wrote {n_calib} calib prompts')

In [8]:
n_calib = 12

In [ ]:
# 5.3 project teacher → student. `-d` omitted → lb auto-picks cuda/mps/cpu.
# -L 128 (max_length) + -B 2 (batch) keeps CPU RAM under 8 GB for activations.
!./lb transfer \
    -T Qwen/Qwen3-4B-Instruct-2507 \
    -S Qwen/Qwen3-0.6B \
    -a runs/teacher_4b/math_sft \
    -C data/calib_math.jsonl \
    -o runs/transferred/math_sft \
    -n {n_calib} -L 128 -B 2

In [6]:
N=50

In [7]:
# 5.4 eval transferred adapter

!./lb eval math500 -a runs/transferred/math_sft -n {N} -o results/math500_transferred.json

2026-05-12 15:03:34.841773: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
╭────────────── lb ──────────────╮
│ device  cuda                   │
│ model   Qwen/Qwen3-0.6B        │
│ cfg     configs/sft_gsm8k.yaml │
│ lora_r  16                     │
│ run     qwen3_06b_sft_gsm8k    │
╰────────────────────────────────╯
╭───────────────────── Traceback (most recent call last) ──────────────────────╮
│ /usr/local/lib/python3.12/dist-packages/peft/config.py:317 in _get_peft_type │
│                                                                              │
│   314 │   │   │   config_file = os.path.join(path, CONFIG_NAME)  

## 6 · Merging scaling law (arXiv:2509.24244)

After training k teacher experts (one per math domain), project them all, merge, eval, fit `L(k) = L∞ + A/(k+b)`. Run only when ≥ 2 transferred adapters exist.

In [ ]:
import glob
transferred = sorted(glob.glob('runs/transferred/*/adapter_model.safetensors'))
transferred = [p.rsplit('/', 1)[0] for p in transferred]
print('available transferred adapters:', transferred)
if len(transferred) >= 2:
    flags = ' '.join(f'-a {p}' for p in transferred[:3])
    !./lb merge -m ties {flags} -o runs/merged_ties
    !./lb eval math500 -a runs/merged_ties -n {N} -o results/scaling/ties.json
else:
    print('skip: need ≥ 2 transferred adapters. Train more teachers in cell 5.')

## 7 · Report

In [ ]:
!./lb report
from IPython.display import Markdown
Markdown(open('results/REPORT.md').read())